In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv('/Users/sashipraneethmuthyala/Desktop/tech-layoffs-sentiment-analysis/data/layoffs_events.csv')
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date')

print(df[['date', 'company', 'layoff_count']])
print(df['date'].min(), "to", df['date'].max())

           date       company  layoff_count
0    2020-03-13   Panda Squad           6.0
1    2020-03-13  HopSkipDrive           8.0
2    2020-03-16      Help.com          16.0
3    2020-03-16       Service           NaN
4    2020-03-16     Inspirato         130.0
...         ...           ...           ...
2464 2026-04-13     StarkWare           NaN
2467 2026-04-15           UKG         950.0
2468 2026-04-15          Snap        1000.0
2466 2026-04-15  Productboard           NaN
2469 2026-04-16    Shutterfly          80.0

[2470 rows x 3 columns]
2020-03-13 00:00:00 to 2026-04-16 00:00:00


In [3]:
import sys
print(sys.executable)

/Users/sashipraneethmuthyala/anaconda3/bin/python


In [4]:
df = df.set_index('date')
print(df.head())

                 company     location  layoff_count pct_workforce  \
date                                                                
2020-03-13   Panda Squad  SF Bay Area           6.0           75%   
2020-03-13  HopSkipDrive  Los Angeles           8.0           10%   
2020-03-16      Help.com       Austin          16.0          100%   
2020-03-16       Service  Los Angeles           NaN          100%   
2020-03-16     Inspirato       Denver         130.0           22%   

                industry                                         source_url  \
date                                                                          
2020-03-13      Consumer  https://twitter.com/danielsing er/status/12385...   
2020-03-13  Transportat…  https://layoffs.fyi/2020/04/02/ hopskipdrive-l...   
2020-03-16       Support                                           LinkedIn   
2020-03-16        Travel  https://techcrunch.com/2020/ 03/16/travel-savi...   
2020-03-16        Travel  https://business

In [6]:
monthly_layoffs = df['layoff_count'].resample('ME').sum()
print(monthly_layoffs.head(10))

date
2020-03-31     9533.0
2020-04-30    15219.0
2020-05-31    13483.0
2020-06-30     4823.0
2020-07-31     6656.0
2020-08-31     1418.0
2020-09-30       47.0
2020-10-31      200.0
2020-11-30      177.0
2020-12-31      252.0
Freq: ME, Name: layoff_count, dtype: float64


In [7]:
rolling_3mo = monthly_layoffs.rolling(window=3).mean()
print(rolling_3mo.head(10))

date
2020-03-31             NaN
2020-04-30             NaN
2020-05-31    12745.000000
2020-06-30    11175.000000
2020-07-31     8320.666667
2020-08-31     4299.000000
2020-09-30     2707.000000
2020-10-31      555.000000
2020-11-30      141.333333
2020-12-31      209.666667
Freq: ME, Name: layoff_count, dtype: float64


In [8]:
df_monthly = monthly_layoffs.to_frame()
df_monthly['prev_month'] = df_monthly['layoff_count'].shift(1)
df_monthly['mom_change'] = df_monthly['layoff_count'] - df_monthly['prev_month']
df_monthly['mom_pct_change'] = df_monthly['layoff_count'].pct_change() * 100

print(df_monthly.head(10))

            layoff_count  prev_month  mom_change  mom_pct_change
date                                                            
2020-03-31        9533.0         NaN         NaN             NaN
2020-04-30       15219.0      9533.0      5686.0       59.645442
2020-05-31       13483.0     15219.0     -1736.0      -11.406794
2020-06-30        4823.0     13483.0     -8660.0      -64.229029
2020-07-31        6656.0      4823.0      1833.0       38.005391
2020-08-31        1418.0      6656.0     -5238.0      -78.695913
2020-09-30          47.0      1418.0     -1371.0      -96.685472
2020-10-31         200.0        47.0       153.0      325.531915
2020-11-30         177.0       200.0       -23.0      -11.500000
2020-12-31         252.0       177.0        75.0       42.372881


In [9]:
full_date_range = pd.date_range(start=monthly_layoffs.index.min(), end=monthly_layoffs.index.max(), freq='ME')
missing_months = full_date_range.difference(monthly_layoffs.index)
print(f"Number of missing months: {len(missing_months)}")
print(missing_months)

Number of missing months: 0
DatetimeIndex([], dtype='datetime64[us]', freq='ME')
